# Qwen2-VL Hard Negative Permutation Reranker

This notebook tests a hard-negative permutation reranker on top of the existing best Qwen2-VL adapter.

Key safeguards in this version:

- `NEGATIVE_RATIOS` is actually used when sampling negative types.
- Duplicate negative permutations are removed per sample.
- Training records are aligned with inference: primary A/B data only uses samples where gold is inside top-K.
- A/B labels train only the single `A` or `B` token, not chat end tokens.
- Quick checkpoint selection deduplicates checkpoints before taking top candidates.
- Candidate cache is stored outside the run directory with a manifest and atomic `.partial` writes.
- Tuning chooses checkpoint/top-K/lambda. Holdout is evaluated only once with the selected setting.


In [ ]:
# 1) Install dependencies, then restart runtime once.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_qwen2vl_reranker_deps_installed")
if not MARKER.exists():
    packages = [
        "transformers>=4.49.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")


In [ ]:
# 2) Setup
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import glob
import hashlib
import itertools
import json
import math
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
import random
import re
import shutil
import time
import zipfile
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = None
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import AutoModelForVision2Seq
except ImportError:
    AutoModelForVision2Seq = None
from peft import PeftModel, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

warnings.filterwarnings("ignore", category=FutureWarning, module="bitsandbytes")
warnings.filterwarnings("ignore", message=".*The following generation flags are not valid.*")
transformers_logging.set_verbosity_error()

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
SAMPLE_SUBMISSION_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

MODEL_REPO_ID = "Qwen/Qwen2-VL-2B-Instruct"
USE_MODELSCOPE_BASE_MODEL = True
DRIVE_MODEL_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-2B-Instruct"
INITIAL_ADAPTER_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_lgt_order_refine_v1/runs/20260714_003635/lgt_order_refine/best_adapter"
SPLIT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/id_splits/qwen2vl_lgt_order_refine_20260714_003635"

OUTPUT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_hard_negative_permutation_reranker_v2"
SHARED_CACHE_ROOT = os.path.join(OUTPUT_ROOT, "shared_candidate_cache")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(OUTPUT_ROOT, "runs", RUN_ID)
OUTPUT_DIR = os.path.join(RUN_ROOT, "reranker")
RECORD_DIR = os.path.join(OUTPUT_DIR, "reranker_records")
EVAL_DIR = os.path.join(OUTPUT_DIR, "eval")
BEST_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "best_adapter")
SUBMIT_PATH = os.path.join(OUTPUT_DIR, "submission.csv")
RUN_SPLIT_DIR = os.path.join(RUN_ROOT, "splits")

SEED = 42
VALID_RATIO = 0.10
QUICK_EVAL_ROWS = 50
TUNING_EVAL_ROWS = 150
HOLDOUT_EVAL_ROWS = 150
TRAIN_CANDIDATE_ROWS = 512  # Pilot cap. Set to None only after speed/recall look sane.
CANDIDATE_CACHE_FLUSH_EVERY = 1
SLOW_CANDIDATE_SECONDS = 180

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

# Defaults are overwritten from the reference best_config/run_config in cell 4.
STRUCTURED_ALPHA = 1.0
STRUCTURED_BETA = 1.0
STRUCTURED_GAMMA = 1.0
PAIRWISE_BIDIRECTIONAL = False
CANDIDATE_TOP_K = 10
TRAIN_TOP_K = 5
RERANK_TOP_K_GRID = [5]
LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]

NEGATIVE_RATIOS = {
    "decoder_top_wrong": 0.40,
    "first_swap": 0.20,
    "last_swap": 0.20,
    "middle_swap": 0.15,
    "random": 0.05,
}
MAX_RECORDS_PER_SAMPLE = 2
BETTER_WRONG_PAIR_RATE = 0.25
BETTER_WRONG_MIN_GAP = 0.25
STRUCTURED_SCORE_COMBINE_MODE = "zscore"
CANDIDATE_PROMPT_VERSION = "qwen2_structured_v1"
RERANKER_PROMPT_VERSION = "ab_candidate_chronology_v2"

LEARNING_RATE = 3e-6
MAX_TRAIN_STEPS = 2200
SAVE_STEPS = 500
LOGGING_STEPS = 20
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4

for directory in [OUTPUT_ROOT, SHARED_CACHE_ROOT, RUN_ROOT, OUTPUT_DIR, RECORD_DIR, EVAL_DIR, BEST_ADAPTER_DIR, RUN_SPLIT_DIR, SPLIT_DIR]:
    os.makedirs(directory, exist_ok=True)

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.exists(TEST_CSV), TEST_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR
assert os.path.isdir(TEST_IMAGE_DIR), TEST_IMAGE_DIR
assert os.path.exists(os.path.join(INITIAL_ADAPTER_DIR, "adapter_config.json")), INITIAL_ADAPTER_DIR

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("shared cache:", SHARED_CACHE_ROOT)
print("initial adapter:", INITIAL_ADAPTER_DIR)


In [ ]:
# 3) IO, split, and data helpers
def save_json(data, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(records, path):
    os.makedirs(os.path.dirname(str(path)), exist_ok=True)
    partial = str(path) + ".partial"
    with open(partial, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(partial, path)


def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def sha16(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16]


def ids_hash(ids):
    return sha16("\n".join(str(value) for value in ids))


def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def sequence_to_answer(order):
    answer = [0] * 4
    for rank, image_number in enumerate(order, start=1):
        answer[int(image_number) - 1] = rank
    return answer


def format_order(order):
    return "[" + ", ".join(str(int(value)) for value in order) + "]"


def row_image_paths(row, image_root):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def rows_by_ids(dataframe, ids):
    ids = [str(value) for value in ids]
    subset = dataframe[dataframe["Id"].isin(ids)].copy()
    order = {sample_id: index for index, sample_id in enumerate(ids)}
    subset["_split_order"] = subset["Id"].map(order)
    return subset.sort_values("_split_order").drop(columns=["_split_order"]).reset_index(drop=True)


def validate_split_ids(split_ids, all_ids):
    all_ids = set(str(value) for value in all_ids)
    for name, ids in split_ids.items():
        ids = [str(value) for value in ids]
        duplicated = pd.Series(ids).value_counts()
        duplicated = duplicated[duplicated > 1]
        assert duplicated.empty, f"{name} has duplicated Ids: {duplicated.head().to_dict()}"
        missing = sorted(set(ids) - all_ids)
        assert not missing, f"{name} has Ids not found in train.csv: {missing[:5]}"
    train_ids = set(split_ids["train_ids.json"])
    validation_ids = set(split_ids["validation_ids.json"])
    quick_ids = set(split_ids["quick50_ids.json"])
    tuning_ids = set(split_ids["tuning150_ids.json"])
    holdout_ids = set(split_ids["holdout150_ids.json"])
    assert train_ids.isdisjoint(validation_ids)
    assert quick_ids <= validation_ids
    assert tuning_ids <= validation_ids
    assert holdout_ids <= validation_ids
    assert quick_ids.isdisjoint(tuning_ids)
    assert quick_ids.isdisjoint(holdout_ids)
    assert tuning_ids.isdisjoint(holdout_ids)


def derive_eval_split_ids(validation_ids):
    validation_shuffle = [str(value) for value in validation_ids]
    rng = np.random.default_rng(SEED)
    rng.shuffle(validation_shuffle)
    quick50_ids = validation_shuffle[:min(QUICK_EVAL_ROWS, len(validation_shuffle))]
    remaining = validation_shuffle[len(quick50_ids):]
    tuning150_ids = remaining[:min(TUNING_EVAL_ROWS, len(remaining))]
    holdout150_ids = remaining[len(tuning150_ids):len(tuning150_ids) + min(HOLDOUT_EVAL_ROWS, max(0, len(remaining) - len(tuning150_ids)))]
    return quick50_ids, tuning150_ids, holdout150_ids


def make_or_load_split_ids(dataframe):
    names = ["train_ids.json", "validation_ids.json", "quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]
    paths = {name: os.path.join(SPLIT_DIR, name) for name in names}
    split_ids = {}
    if os.path.exists(paths["train_ids.json"]) and os.path.exists(paths["validation_ids.json"]):
        split_ids["train_ids.json"] = [str(value) for value in load_json(paths["train_ids.json"])]
        split_ids["validation_ids.json"] = [str(value) for value in load_json(paths["validation_ids.json"])]
    else:
        unique_ids = dataframe["Id"].unique().copy()
        rng = np.random.default_rng(SEED)
        rng.shuffle(unique_ids)
        valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
        split_ids["validation_ids.json"] = [str(value) for value in unique_ids[:valid_size]]
        split_ids["train_ids.json"] = [str(value) for value in unique_ids[valid_size:]]
        save_json(split_ids["train_ids.json"], paths["train_ids.json"])
        save_json(split_ids["validation_ids.json"], paths["validation_ids.json"])

    quick_ids, tuning_ids, holdout_ids = derive_eval_split_ids(split_ids["validation_ids.json"])
    derived = {"quick50_ids.json": quick_ids, "tuning150_ids.json": tuning_ids, "holdout150_ids.json": holdout_ids}
    for name in ["quick50_ids.json", "tuning150_ids.json", "holdout150_ids.json"]:
        if os.path.exists(paths[name]):
            split_ids[name] = [str(value) for value in load_json(paths[name])]
        else:
            split_ids[name] = derived[name]
            save_json(split_ids[name], paths[name])

    validate_split_ids(split_ids, dataframe["Id"].astype(str).tolist())
    for name, ids in split_ids.items():
        save_json(ids, os.path.join(RUN_SPLIT_DIR, name))
    return split_ids


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV) if os.path.exists(SAMPLE_SUBMISSION_CSV) else None
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_ids = make_or_load_split_ids(train_df)
training_df = rows_by_ids(train_df, split_ids["train_ids.json"])
validation_df = rows_by_ids(train_df, split_ids["validation_ids.json"])
quick50_df = rows_by_ids(train_df, split_ids["quick50_ids.json"])
tuning150_df = rows_by_ids(train_df, split_ids["tuning150_ids.json"])
holdout150_df = rows_by_ids(train_df, split_ids["holdout150_ids.json"])
assert len(training_df) == 8582, len(training_df)
assert len(validation_df) == 953, len(validation_df)
print("train/validation/quick/tuning/holdout:", len(training_df), len(validation_df), len(quick50_df), len(tuning150_df), len(holdout150_df))


In [ ]:
# 4) Load and validate the reference structured-decoder config
def find_reference_run_config():
    candidates = [
        os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(INITIAL_ADAPTER_DIR))), "run_config.json"),
        os.path.join(os.path.dirname(os.path.dirname(INITIAL_ADAPTER_DIR)), "run_config.json"),
        os.path.join(os.path.dirname(INITIAL_ADAPTER_DIR), "run_config.json"),
        os.path.join(INITIAL_ADAPTER_DIR, "run_config.json"),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def nested_get(mapping, path, default=None):
    current = mapping if isinstance(mapping, dict) else {}
    for key in path:
        if not isinstance(current, dict) or key not in current:
            return default
        current = current[key]
    return current


def first_not_none(*values):
    for value in values:
        if value is not None:
            return value
    return None


def sha16_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()[:16]


def adapter_weight_fingerprint(adapter_dir):
    files = []
    for name in ["adapter_model.safetensors", "adapter_model.bin", "pytorch_model.bin"]:
        weight_path = os.path.join(adapter_dir, name)
        if os.path.exists(weight_path):
            stat = os.stat(weight_path)
            files.append({
                "name": name,
                "size": int(stat.st_size),
                "mtime_ns": int(stat.st_mtime_ns),
                "sha16": sha16_file(weight_path),
            })
    return {
        "adapter_dir": adapter_dir,
        "adapter_config_hash": sha16(load_json(os.path.join(adapter_dir, "adapter_config.json"))),
        "weight_files": files,
    }


INITIAL_BEST_CONFIG_PATH = os.path.join(INITIAL_ADAPTER_DIR, "best_config.json")
assert os.path.exists(INITIAL_BEST_CONFIG_PATH), INITIAL_BEST_CONFIG_PATH
initial_best_config = load_json(INITIAL_BEST_CONFIG_PATH)
reference_run_config_path = find_reference_run_config()
reference_run_config = load_json(reference_run_config_path) if reference_run_config_path else {}
reference_candidate_config = first_not_none(
    initial_best_config.get("candidate_generator"),
    reference_run_config.get("candidate_generator"),
    {},
)
reference_decoding_config = first_not_none(
    initial_best_config.get("decoding"),
    reference_run_config.get("decoding"),
    {},
)

print("reference best_config:", INITIAL_BEST_CONFIG_PATH)
print("reference run_config:", reference_run_config_path)
print(json.dumps(initial_best_config, ensure_ascii=False, indent=2)[:2000])

STRUCTURED_ALPHA = float(first_not_none(
    nested_get(reference_candidate_config, ["alpha"]),
    nested_get(reference_decoding_config, ["alpha"]),
    initial_best_config.get("alpha"),
    reference_run_config.get("alpha"),
    STRUCTURED_ALPHA,
))
STRUCTURED_BETA = float(first_not_none(
    nested_get(reference_candidate_config, ["beta"]),
    nested_get(reference_decoding_config, ["beta"]),
    initial_best_config.get("beta"),
    reference_run_config.get("beta"),
    STRUCTURED_BETA,
))
STRUCTURED_GAMMA = float(first_not_none(
    nested_get(reference_candidate_config, ["gamma"]),
    nested_get(reference_decoding_config, ["gamma"]),
    initial_best_config.get("gamma"),
    reference_run_config.get("gamma"),
    STRUCTURED_GAMMA,
))

PAIRWISE_BIDIRECTIONAL = bool(first_not_none(
    nested_get(reference_candidate_config, ["pairwise_bidirectional"]),
    initial_best_config.get("pairwise_bidirectional"),
    reference_run_config.get("pairwise_bidirectional"),
    PAIRWISE_BIDIRECTIONAL,
))

MIN_PIXELS = int(first_not_none(
    nested_get(reference_candidate_config, ["min_pixels"]),
    reference_run_config.get("min_pixels"),
    MIN_PIXELS,
))
MAX_PIXELS = int(first_not_none(
    nested_get(reference_candidate_config, ["max_pixels"]),
    reference_run_config.get("max_pixels"),
    MAX_PIXELS,
))

if "model_repo_id" in reference_run_config:
    assert "Qwen2" in str(reference_run_config["model_repo_id"]) or "Qwen2" in MODEL_REPO_ID, reference_run_config["model_repo_id"]
if "task_ratios" in reference_run_config:
    assert set(reference_run_config["task_ratios"]) >= {"order", "pairwise", "first", "last"}, reference_run_config["task_ratios"]

reference_config_summary = {
    "best_config_path": INITIAL_BEST_CONFIG_PATH,
    "run_config_path": reference_run_config_path,
    "alpha": STRUCTURED_ALPHA,
    "beta": STRUCTURED_BETA,
    "gamma": STRUCTURED_GAMMA,
    "pairwise_bidirectional": PAIRWISE_BIDIRECTIONAL,
    "min_pixels": MIN_PIXELS,
    "max_pixels": MAX_PIXELS,
    "model_repo_id": MODEL_REPO_ID,
    "split_hashes": {name: ids_hash(ids) for name, ids in split_ids.items()},
    "adapter_fingerprint": adapter_weight_fingerprint(INITIAL_ADAPTER_DIR),
    "candidate_prompt_version": CANDIDATE_PROMPT_VERSION,
    "reranker_prompt_version": RERANKER_PROMPT_VERSION,
}
save_json(reference_config_summary, os.path.join(OUTPUT_DIR, "reference_config_summary.json"))
print("reference config summary:")
print(json.dumps(reference_config_summary, ensure_ascii=False, indent=2))


In [ ]:
# 5) Model loading and structured candidate generator
def model_cache_is_complete(model_dir):
    if not os.path.exists(os.path.join(model_dir, "config.json")):
        return False
    has_weight = any(
        os.path.exists(os.path.join(model_dir, name))
        for name in ["model.safetensors.index.json", "pytorch_model.bin", "pytorch_model.bin.index.json"]
    ) or bool(glob.glob(os.path.join(model_dir, "*.safetensors")))
    has_processor = any(
        os.path.exists(os.path.join(model_dir, name))
        for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"]
    )
    return bool(has_weight and has_processor)


def copy_model_snapshot_to_drive(snapshot_dir, drive_dir):
    os.makedirs(os.path.dirname(drive_dir), exist_ok=True)
    tmp_dir = drive_dir + ".tmp"
    if os.path.exists(tmp_dir):
        shutil.rmtree(tmp_dir)
    shutil.copytree(snapshot_dir, tmp_dir, dirs_exist_ok=True)
    if not model_cache_is_complete(tmp_dir):
        raise RuntimeError(f"Downloaded model snapshot is incomplete: {tmp_dir}")
    if os.path.exists(drive_dir):
        backup_dir = drive_dir + ".incomplete"
        if os.path.exists(backup_dir):
            shutil.rmtree(backup_dir)
        shutil.move(drive_dir, backup_dir)
        print("Existing incomplete model cache moved to:", backup_dir)
    os.replace(tmp_dir, drive_dir)
    return drive_dir


def ensure_base_model_path():
    if model_cache_is_complete(DRIVE_MODEL_DIR):
        print("Using complete cached base model:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    if os.path.exists(DRIVE_MODEL_DIR):
        print("Cached base model exists but looks incomplete; redownloading:", DRIVE_MODEL_DIR)

    if not USE_MODELSCOPE_BASE_MODEL:
        print("Using Hugging Face repo id directly:", MODEL_REPO_ID)
        return MODEL_REPO_ID

    os.makedirs(os.path.dirname(DRIVE_MODEL_DIR), exist_ok=True)
    errors = []

    try:
        print("Downloading base model via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        snapshot_dir = modelscope_snapshot_download(
            MODEL_REPO_ID,
            cache_dir="/content/modelscope_cache",
        )
        return copy_model_snapshot_to_drive(snapshot_dir, DRIVE_MODEL_DIR)
    except Exception as exc:
        errors.append(("modelscope", repr(exc)))
        print("ModelScope download failed; trying Hugging Face snapshot_download.")
        print(errors[-1])

    try:
        from huggingface_hub import snapshot_download as hf_snapshot_download
        snapshot_dir = hf_snapshot_download(
            repo_id=MODEL_REPO_ID,
            local_dir="/content/hf_model_cache/Qwen2-VL-2B-Instruct",
            local_dir_use_symlinks=False,
            resume_download=True,
        )
        return copy_model_snapshot_to_drive(snapshot_dir, DRIVE_MODEL_DIR)
    except Exception as exc:
        errors.append(("huggingface", repr(exc)))
        print("Hugging Face download failed.")
        print(errors[-1])

    if os.path.exists(os.path.join(DRIVE_MODEL_DIR, "config.json")):
        print("Falling back to existing Drive cache despite incomplete-check failure:", DRIVE_MODEL_DIR)
        return DRIVE_MODEL_DIR
    raise RuntimeError(f"Could not prepare base model cache. Errors: {errors}")


def load_model_class():
    if Qwen2VLForConditionalGeneration is not None:
        return Qwen2VLForConditionalGeneration
    if AutoModelForImageTextToText is not None:
        return AutoModelForImageTextToText
    if AutoModelForVision2Seq is not None:
        return AutoModelForVision2Seq
    raise ImportError("No compatible Qwen2-VL model class found.")


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = os.path.isdir(MODEL_ID)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def model_device(active_model):
    return next(active_model.parameters()).device


def load_base_model(for_training=False):
    model = load_model_class().from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
        local_files_only=MODEL_LOCAL_FILES_ONLY,
        trust_remote_code=True,
    )
    model.config.use_cache = False
    if for_training:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model


def load_adapter_model(adapter_dir, is_trainable=False):
    base = load_base_model(for_training=is_trainable)
    model = PeftModel.from_pretrained(base, adapter_dir, is_trainable=is_trainable)
    model.config.use_cache = False
    if not is_trainable:
        model.eval()
    if hasattr(model, "generation_config"):
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None
        model.generation_config.num_beams = 1
    return model


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}; this notebook expects single-token labels.")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
AB_TOKEN_IDS = {"A": single_token_id("A"), "B": single_token_id("B")}
print("digit token ids:", DIGIT_TOKEN_IDS)
print("A/B token ids:", AB_TOKEN_IDS)


def original_task_instruction(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return (
            f"Caption:\n{sentence}\n\n"
            "Question: Which image occurs first?\n"
            "If the first image occurs earlier, answer 1.\n"
            "If the second image occurs earlier, answer 2.\n"
            "Answer only 1 or 2."
        )
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the beginning of the story?\nAnswer only the image number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nQuestion: Which image represents the end of the story?\nAnswer only the image number from 1 to 4."
    raise ValueError(task_type)


def original_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + original_task_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


def make_original_eval_example(row, task_type, image_root, pair=None):
    image_paths = row_image_paths(row, image_root)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    return example


In [ ]:
# 5) Candidate cache generation
@torch.no_grad()
def score_digit_candidate_batch(active_model, examples, candidates_per_example):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        texts = [
            processor.apply_chat_template(original_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
            for example in examples
        ]
        images = [[load_rgb(path) for path in example["image_paths"]] for example in examples]
        inputs = processor(text=texts, images=images, padding=True, return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        result = []
        for row_index, candidates in enumerate(candidates_per_example):
            last_pos = int(inputs["attention_mask"][row_index].sum().item()) - 1
            logits = outputs.logits[row_index, last_pos]
            token_ids = [DIGIT_TOKEN_IDS[int(candidate)] for candidate in candidates]
            probs = torch.softmax(logits[token_ids].float(), dim=-1).detach().cpu().numpy()
            result.append({int(candidate): float(prob) for candidate, prob in zip(candidates, probs)})
        return result
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def score_digit_candidates(active_model, example, candidates):
    return score_digit_candidate_batch(active_model, [example], [candidates])[0]


def structured_score(first_probs, last_probs, pair_probs, order, alpha, beta, gamma):
    eps = 1e-12
    pair_score = np.mean([
        math.log(float(pair_probs[f"{order[i]}>{order[j]}"]) + eps)
        for i in range(4)
        for j in range(i + 1, 4)
    ])
    first_score = math.log(float(first_probs[str(order[0])]) + eps)
    last_score = math.log(float(last_probs[str(order[-1])]) + eps)
    return float(alpha * pair_score + beta * first_score + gamma * last_score)


def order_metric_row(pred_order, gold_order):
    pred_ranks = {int(image_number): position for position, image_number in enumerate(pred_order)}
    gold_ranks = {int(image_number): position for position, image_number in enumerate(gold_order)}
    pair_accuracy = np.mean([
        (pred_ranks[a] < pred_ranks[b]) == (gold_ranks[a] < gold_ranks[b])
        for a, b in itertools.combinations([1, 2, 3, 4], 2)
    ])
    return {
        "exact_match": float(list(pred_order) == list(gold_order)),
        "pair_accuracy": float(pair_accuracy),
        "position_accuracy": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
    }


def build_candidate_record(active_model, row, image_root, has_gold=True, top_k=CANDIDATE_TOP_K):
    examples = [
        make_original_eval_example(row, "first", image_root),
        make_original_eval_example(row, "last", image_root),
    ]
    candidates = [[1, 2, 3, 4], [1, 2, 3, 4]]
    pair_specs = []
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        pair_specs.append((a, b, False))
        examples.append(make_original_eval_example(row, "pairwise", image_root, pair=(a, b)))
        candidates.append([1, 2])
        if PAIRWISE_BIDIRECTIONAL:
            pair_specs.append((a, b, True))
            examples.append(make_original_eval_example(row, "pairwise", image_root, pair=(b, a)))
            candidates.append([1, 2])

    batch_probs = score_digit_candidate_batch(active_model, examples, candidates)
    first_probs = {str(k): v for k, v in batch_probs[0].items()}
    last_probs = {str(k): v for k, v in batch_probs[1].items()}

    pair_probs = {}
    offset = 2
    for first_index, second_index in PAIR_INDICES:
        a, b = first_index + 1, second_index + 1
        forward = batch_probs[offset]
        offset += 1
        p_a_before_b = float(forward[1])
        if PAIRWISE_BIDIRECTIONAL:
            reverse = batch_probs[offset]
            offset += 1
            p_a_before_b = 0.5 * (p_a_before_b + float(reverse[2]))
        pair_probs[f"{a}>{b}"] = float(p_a_before_b)
        pair_probs[f"{b}>{a}"] = float(1.0 - p_a_before_b)

    scored = []
    for order in PERMUTATIONS:
        score = structured_score(
            first_probs,
            last_probs,
            pair_probs,
            order,
            STRUCTURED_ALPHA,
            STRUCTURED_BETA,
            STRUCTURED_GAMMA,
        )
        scored.append({"order": list(order), "structured_score": score})
    scored = sorted(scored, key=lambda item: item["structured_score"], reverse=True)
    for rank, item in enumerate(scored, start=1):
        item["rank"] = rank

    gold_order = None
    gold_rank = None
    if has_gold:
        gold_order = order_to_sequence([int(value) for value in row["Answer_list"]])
        for item in scored:
            item["is_gold"] = item["order"] == gold_order
            if item["is_gold"]:
                gold_rank = item["rank"]
    else:
        for item in scored:
            item["is_gold"] = None

    return {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "gold_order": gold_order,
        "gold_rank": gold_rank,
        "candidate_orders": [item["order"] for item in scored[:top_k]],
        "structured_scores": [float(item["structured_score"]) for item in scored[:top_k]],
        "all_candidates": scored,
    }


def candidate_cache_manifest(split_name, rows, image_root):
    return {
        "split": split_name,
        "model_repo_id": MODEL_REPO_ID,
        "initial_adapter": INITIAL_ADAPTER_DIR,
        "adapter_fingerprint": adapter_weight_fingerprint(INITIAL_ADAPTER_DIR),
        "ids_hash": ids_hash(rows["Id"].astype(str).tolist()),
        "row_count": int(len(rows)),
        "candidate_top_k": CANDIDATE_TOP_K,
        "alpha": STRUCTURED_ALPHA,
        "beta": STRUCTURED_BETA,
        "gamma": STRUCTURED_GAMMA,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
        "bidirectional_pairwise": PAIRWISE_BIDIRECTIONAL,
        "candidate_prompt_version": CANDIDATE_PROMPT_VERSION,
        "batched_candidate_scoring": True,
        "image_root_name": os.path.basename(image_root),
    }


def candidate_cache_paths(split_name, rows, image_root):
    manifest = candidate_cache_manifest(split_name, rows, image_root)
    key = sha16(json.dumps(manifest, sort_keys=True, ensure_ascii=False))
    cache_dir = os.path.join(SHARED_CACHE_ROOT, key)
    return cache_dir, os.path.join(cache_dir, f"{split_name}_candidates.jsonl"), os.path.join(cache_dir, "manifest.json"), manifest


def append_jsonl(record, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())


def generate_candidate_cache(split_name, rows, image_root, has_gold=True, force=False):
    cache_dir, path, manifest_path, manifest = candidate_cache_paths(split_name, rows, image_root)
    os.makedirs(cache_dir, exist_ok=True)
    expected_ids = rows["Id"].astype(str).tolist()
    partial_path = path + ".partial"

    if os.path.exists(path) and os.path.exists(manifest_path) and not force:
        cached = read_jsonl(path)
        cached_ids = [record["sample_id"] for record in cached]
        if cached_ids == expected_ids and load_json(manifest_path) == manifest:
            print("[CACHE]", path)
            return cached
        print("Incomplete or stale cache. Regenerating:", path)

    completed = []
    if os.path.exists(partial_path) and os.path.exists(manifest_path) and load_json(manifest_path) == manifest and not force:
        completed = read_jsonl(partial_path)
        completed_ids = [record["sample_id"] for record in completed]
        if expected_ids[:len(completed_ids)] == completed_ids:
            print("[RESUME]", partial_path, len(completed))
        else:
            completed = []

    if force or not completed:
        if os.path.exists(partial_path):
            os.remove(partial_path)
        save_json(manifest, manifest_path)

    completed_ids = {record["sample_id"] for record in completed}
    records = list(completed)
    remaining = rows[~rows["Id"].astype(str).isin(completed_ids)].copy()

    model = load_adapter_model(INITIAL_ADAPTER_DIR, is_trainable=False)
    try:
        if torch.cuda.is_available():
            print("candidate cache GPU:", torch.cuda.get_device_name(0))
        print("candidate cache rows:", split_name, len(rows), "remaining:", len(remaining))
        for index, (_, row) in enumerate(tqdm(remaining.iterrows(), total=len(remaining), desc=f"candidate cache: {split_name}"), start=len(records) + 1):
            started = time.time()
            record = build_candidate_record(model, row, image_root, has_gold=has_gold)
            elapsed = time.time() - started
            records.append(record)
            append_jsonl(record, partial_path)
            if elapsed > SLOW_CANDIDATE_SECONDS:
                print(f"WARNING: one candidate record took {elapsed:.1f}s. Check GPU runtime/device_map before continuing full cache.")
            if index == 1:
                print(f"first candidate seconds: {elapsed:.1f}")
        os.replace(partial_path, path)
        save_json(manifest, manifest_path)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()
    print("saved:", path, len(records))
    return records


def topk_recall(records, ks=(1, 3, 5, 10, 24)):
    rows = []
    for k in ks:
        hits = []
        for record in records:
            gold = record.get("gold_order")
            if gold is None:
                continue
            candidates = [item["order"] for item in record["all_candidates"][:k]]
            hits.append(int(gold in candidates))
        rows.append({"k": k, "gold_recall": float(np.mean(hits)) if hits else np.nan})
    return pd.DataFrame(rows)


In [ ]:
# 6) Generate or load candidate caches and inspect recall.
if TRAIN_CANDIDATE_ROWS is None:
    train_candidate_df = training_df.copy()
else:
    train_candidate_df = training_df.head(min(TRAIN_CANDIDATE_ROWS, len(training_df))).copy()
print("train candidate rows:", len(train_candidate_df), "/", len(training_df))

quick_candidates = generate_candidate_cache("quick50", quick50_df, TRAIN_IMAGE_DIR, has_gold=True)
tuning_candidates = generate_candidate_cache("tuning150", tuning150_df, TRAIN_IMAGE_DIR, has_gold=True)
holdout_candidates = generate_candidate_cache("holdout150", holdout150_df, TRAIN_IMAGE_DIR, has_gold=True)
train_candidates = generate_candidate_cache(f"train{len(train_candidate_df)}", train_candidate_df, TRAIN_IMAGE_DIR, has_gold=True)

recall_tables = []
for split_name, records in [
    ("train", train_candidates),
    ("quick50", quick_candidates),
    ("tuning150", tuning_candidates),
    ("holdout150", holdout_candidates),
]:
    df = topk_recall(records)
    df.insert(0, "split", split_name)
    recall_tables.append(df)
recall_df = pd.concat(recall_tables, ignore_index=True)
recall_df.to_csv(os.path.join(EVAL_DIR, "topk_recall.csv"), index=False)
display(recall_df)

train_recall_at_k = float(recall_df[(recall_df["split"].eq("train")) & (recall_df["k"].eq(TRAIN_TOP_K))]["gold_recall"].iloc[0])
print(f"Train recall@{TRAIN_TOP_K}:", train_recall_at_k)
if train_recall_at_k < 0.65:
    print("WARNING: low recall@K limits reranker upside. Consider increasing TRAIN_TOP_K/CANDIDATE_TOP_K.")


In [ ]:
# 7) Hard negative and preference-record generation
def candidate_quality(order, gold):
    metric = order_metric_row(order, gold)
    return 6.0 * metric["exact_match"] + 2.0 * metric["pair_accuracy"] + 1.0 * metric["position_accuracy"]


def structured_score_by_order(record):
    return {tuple(item["order"]): float(item["structured_score"]) for item in record["all_candidates"]}


def dedupe_available_negatives(record, rng, top_k=TRAIN_TOP_K):
    gold = list(record["gold_order"])
    top_orders = [list(order) for order in record["candidate_orders"][:top_k]]
    wrong_top = [order for order in top_orders if order != gold]
    available = {}
    seen = set()
    duplicate_removed = 0
    unavailable_types = []

    def add(name, order):
        nonlocal duplicate_removed
        if order is None or list(order) == gold:
            unavailable_types.append(name)
            return False
        if list(order) not in top_orders:
            unavailable_types.append(name)
            return False
        key = tuple(order)
        if key in seen:
            duplicate_removed += 1
            return False
        seen.add(key)
        available[name] = list(order)
        return True

    add("decoder_top_wrong", wrong_top[0] if wrong_top else None)
    add("first_swap", next((order for order in wrong_top if order[0] != gold[0]), None))
    add("last_swap", next((order for order in wrong_top if order[-1] != gold[-1]), None))
    add("middle_swap", next((order for order in wrong_top if order[0] == gold[0] and order[-1] == gold[-1] and order[1:3] != gold[1:3]), None))

    remaining = [order for order in wrong_top if tuple(order) not in seen]
    random_order = remaining[int(rng.integers(0, len(remaining)))] if remaining else None
    add("random", random_order)
    return available, {
        "duplicate_removed": duplicate_removed,
        "unavailable_types": unavailable_types,
        "wrong_top_count": len(wrong_top),
    }


def make_ab_record(record, order_a, order_b, target, negative_type, comparison_type):
    if list(order_a) == list(order_b):
        return None
    score_map = structured_score_by_order(record)
    score_a = float(score_map[tuple(order_a)])
    score_b = float(score_map[tuple(order_b)])
    preferred_order = list(order_a) if target == "A" else list(order_b)
    rejected_order = list(order_b) if target == "A" else list(order_a)
    preferred_score = score_a if target == "A" else score_b
    rejected_score = score_b if target == "A" else score_a
    return {
        "sample_id": record["sample_id"],
        "sentence": record["sentence"],
        "gold_order": list(record["gold_order"]),
        "candidate_a": list(order_a),
        "candidate_b": list(order_b),
        "target": target,
        "negative_type": negative_type,
        "comparison_type": comparison_type,
        "gold_rank": record.get("gold_rank"),
        "candidate_a_structured_score": score_a,
        "candidate_b_structured_score": score_b,
        "preferred_structured_score": preferred_score,
        "rejected_structured_score": rejected_score,
        "structured_gap_abs": abs(score_a - score_b),
        "gold_structured_score": float(score_map[tuple(record["gold_order"])]),
        "negative_structured_score": rejected_score if comparison_type == "gold_vs_hard_wrong" else None,
        "preferred_order": preferred_order,
        "rejected_order": rejected_order,
        "negative_in_topk": True,
    }


def weighted_negative_types(available, count, rng):
    names = list(available)
    if not names:
        return []
    probs = np.array([NEGATIVE_RATIOS[name] for name in names], dtype=np.float64)
    probs = probs / probs.sum()
    count = min(count, len(names))
    return list(rng.choice(names, size=count, replace=False, p=probs))


def build_reranker_records(candidate_records, split_name):
    rng = np.random.default_rng(SEED + sum(ord(ch) for ch in split_name))
    records = []
    skipped_gold_outside_topk = 0
    skipped_no_negative = 0
    duplicate_removed = 0
    unavailable_counter = {}
    gold_rank_values = []
    wrong_top_counts = []

    for record in candidate_records:
        gold = list(record["gold_order"])
        gold_rank_values.append(record.get("gold_rank"))
        top_orders = [list(order) for order in record["candidate_orders"][:TRAIN_TOP_K]]
        if gold not in top_orders:
            skipped_gold_outside_topk += 1
            continue

        available, info = dedupe_available_negatives(record, rng, top_k=TRAIN_TOP_K)
        duplicate_removed += info["duplicate_removed"]
        wrong_top_counts.append(info["wrong_top_count"])
        for name in info["unavailable_types"]:
            unavailable_counter[name] = unavailable_counter.get(name, 0) + 1
        if not available:
            skipped_no_negative += 1
            continue

        selected_types = weighted_negative_types(available, max(1, MAX_RECORDS_PER_SAMPLE - 1), rng)
        for negative_type in selected_types:
            negative = available[negative_type]
            if rng.random() < 0.5:
                records.append(make_ab_record(record, gold, negative, "A", negative_type, "gold_vs_hard_wrong"))
            else:
                records.append(make_ab_record(record, negative, gold, "B", negative_type, "gold_vs_hard_wrong"))

        wrong_top = [order for order in top_orders if order != gold]
        if len(wrong_top) >= 2 and rng.random() < BETTER_WRONG_PAIR_RATE:
            pairs = []
            for order_a, order_b in itertools.combinations(wrong_top, 2):
                qa = candidate_quality(order_a, gold)
                qb = candidate_quality(order_b, gold)
                if abs(qa - qb) >= BETTER_WRONG_MIN_GAP:
                    pairs.append((order_a, order_b, qa, qb))
            if pairs:
                order_a, order_b, qa, qb = pairs[int(rng.integers(0, len(pairs)))]
                target = "A" if qa > qb else "B"
                records.append(make_ab_record(record, order_a, order_b, target, "better_wrong", "better_wrong_vs_worse_wrong"))

    records = [record for record in records if record is not None]
    records = rebalance_targets(records, SEED + 1000 + sum(ord(ch) for ch in split_name))
    target_distribution = pd.Series([record["target"] for record in records]).value_counts().to_dict()
    negative_distribution = pd.Series([record["negative_type"] for record in records]).value_counts().to_dict()
    comparison_distribution = pd.Series([record["comparison_type"] for record in records]).value_counts().to_dict()
    gold_rank_distribution = pd.Series(gold_rank_values).value_counts(dropna=False).sort_index().to_dict()
    stats = {
        "split": split_name,
        "records": len(records),
        "source_samples": len(candidate_records),
        "skipped_gold_outside_topk": skipped_gold_outside_topk,
        "skipped_no_negative": skipped_no_negative,
        "duplicate_removed": duplicate_removed,
        "unavailable_negative_types": unavailable_counter,
        "target_distribution": target_distribution,
        "negative_type_distribution": negative_distribution,
        "comparison_type_distribution": comparison_distribution,
        "gold_rank_distribution": gold_rank_distribution,
        "negative_in_topk_rate": float(np.mean([record.get("negative_in_topk", False) for record in records])) if records else np.nan,
        "wrong_top_count_mean": float(np.mean(wrong_top_counts)) if wrong_top_counts else np.nan,
    }
    print(json.dumps(stats, ensure_ascii=False, indent=2))
    if split_name == "train":
        effective_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
        steps_per_epoch = math.ceil(len(records) / max(effective_batch, 1)) if records else 0
        estimated_epochs = float(MAX_TRAIN_STEPS / steps_per_epoch) if MAX_TRAIN_STEPS > 0 and steps_per_epoch else np.nan
        print({
            "train_records": len(records),
            "effective_batch": effective_batch,
            "steps_per_epoch": steps_per_epoch,
            "max_train_steps": MAX_TRAIN_STEPS,
            "estimated_epochs": estimated_epochs,
        })
    assert negative_distribution.get("random", 0) > 0 or split_name != "train", negative_distribution
    return records, stats


def rebalance_targets(records, seed):
    rng = np.random.default_rng(seed)
    records = [dict(record) for record in records]
    rng.shuffle(records)
    a_records = [record for record in records if record["target"] == "A"]
    b_records = [record for record in records if record["target"] == "B"]
    target = min(len(a_records), len(b_records))
    if target == 0:
        raise ValueError("Cannot balance A/B targets.")
    balanced = a_records[:target] + b_records[:target]
    rng.shuffle(balanced)
    target_counts = pd.Series([record["target"] for record in balanced]).value_counts().to_dict()
    total = len(balanced)
    assert abs(target_counts.get("A", 0) / total - 0.5) < 0.02, target_counts
    assert abs(target_counts.get("B", 0) / total - 0.5) < 0.02, target_counts
    return balanced


def save_records(split_name, records, stats):
    path = os.path.join(RECORD_DIR, f"{split_name}_reranker_records.jsonl")
    write_jsonl(records, path)
    save_json(stats, os.path.join(RECORD_DIR, f"{split_name}_distribution.json"))
    return path


train_reranker_records, train_stats = build_reranker_records(train_candidates, "train")
quick_reranker_records, quick_stats = build_reranker_records(quick_candidates, "quick50")
tuning_reranker_records, tuning_stats = build_reranker_records(tuning_candidates, "tuning150")
holdout_reranker_records, holdout_stats = build_reranker_records(holdout_candidates, "holdout150")

for split_name, records, stats in [
    ("train", train_reranker_records, train_stats),
    ("quick50", quick_reranker_records, quick_stats),
    ("tuning150", tuning_reranker_records, tuning_stats),
    ("holdout150", holdout_reranker_records, holdout_stats),
]:
    print(split_name, save_records(split_name, records, stats))


In [ ]:
# 8) Reranker prompt, dataset, collator
TRAIN_ROW_BY_ID = train_df.assign(Id=train_df["Id"].astype(str)).set_index("Id")
TEST_ROW_BY_ID = test_df.assign(Id=test_df["Id"].astype(str)).set_index("Id")


def row_for_sample(sample_id, image_root):
    return TEST_ROW_BY_ID.loc[str(sample_id)] if image_root == TEST_IMAGE_DIR else TRAIN_ROW_BY_ID.loc[str(sample_id)]


def reranker_instruction(example):
    return (
        "The four images are shuffled frames from one video.\n\n"
        f"Caption:\n{example['sentence']}\n\n"
        f"Candidate A:\n{format_order(example['candidate_a'])}\n\n"
        f"Candidate B:\n{format_order(example['candidate_b'])}\n\n"
        "Which candidate better represents the complete chronological order of the video?\n\n"
        "Each candidate lists the image numbers from earliest to latest.\n"
        "Compare the full temporal progression across all four scenes.\n"
        "Use the visual state changes, actions, objects, and the caption context.\n\n"
        "Answer only A or B."
    )


def reranker_messages(example, image_root, include_answer=False):
    sample_id = str(example["sample_id"])
    row = row_for_sample(sample_id, image_root)
    image_paths = [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]
    content = []
    for idx, _ in enumerate(image_paths, start=1):
        content.append({"type": "text", "text": f"\nImage {idx}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + reranker_instruction(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages, image_paths


class RerankerDataset(Dataset):
    def __init__(self, records, image_root):
        self.records = list(records)
        self.image_root = image_root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = dict(self.records[index])
        item["image_root"] = self.image_root
        return item


class RerankerCollator:
    def __init__(self, processor):
        self.processor = processor
        self.tokenizer = processor.tokenizer
        self.assistant_prefix_ids = self.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

    def _mask_prompt(self, input_ids, target):
        ids = input_ids.tolist()
        labels = torch.full_like(input_ids, -100)
        prefix = self.assistant_prefix_ids
        start = None
        for i in range(0, max(0, len(ids) - len(prefix) + 1)):
            if ids[i:i + len(prefix)] == prefix:
                start = i + len(prefix)
        if start is None:
            tail = self.tokenizer.decode(ids[-160:], skip_special_tokens=False)
            raise ValueError(f"Assistant prefix not found. target={target!r}, tail={tail!r}")

        expected = AB_TOKEN_IDS[str(target)]
        if ids[start] != expected:
            tail = self.tokenizer.decode(ids[start:start + 10], skip_special_tokens=False)
            raise ValueError(f"Unexpected target token. target={target!r}, expected={expected}, actual={ids[start]}, tail={tail!r}")
        labels[start] = expected
        return labels

    def __call__(self, batch):
        texts, images, targets, negative_types = [], [], [], []
        for example in batch:
            messages, image_paths = reranker_messages(example, example["image_root"], include_answer=True)
            texts.append(self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
            images.append([load_rgb(path) for path in image_paths])
            targets.append(str(example["target"]))
            negative_types.append(example.get("negative_type", "unknown"))
        encoded = self.processor(text=texts, images=images, padding=True, return_tensors="pt")
        labels = torch.stack([self._mask_prompt(row, target) for row, target in zip(encoded["input_ids"], targets)])
        encoded["labels"] = labels
        encoded["negative_type"] = negative_types
        return encoded


_mask_check = RerankerCollator(processor)([dict(record, image_root=TRAIN_IMAGE_DIR) for record in train_reranker_records[:8]])
print("reranker label mask token counts:", _mask_check["labels"].ne(-100).sum(dim=1).tolist())
del _mask_check


In [ ]:
# 9) Train preference-classification reranker from the existing best adapter
class RerankerTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs.pop("negative_type", None)
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        token_losses = torch.nn.functional.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            reduction="none",
            ignore_index=-100,
        ).view_as(shift_labels)
        mask = shift_labels.ne(-100)
        sample_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        loss = sample_losses.mean()
        return (loss, outputs) if return_outputs else loss


model = load_adapter_model(INITIAL_ADAPTER_DIR, is_trainable=True)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=MAX_TRAIN_STEPS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    logging_steps=LOGGING_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=5,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=SEED,
    data_seed=SEED,
)

trainer = RerankerTrainer(
    model=model,
    args=training_args,
    train_dataset=RerankerDataset(train_reranker_records, TRAIN_IMAGE_DIR),
    data_collator=RerankerCollator(processor),
)
trainer.train()
final_adapter_dir = os.path.join(OUTPUT_DIR, "final_adapter")
model.save_pretrained(final_adapter_dir)
processor.save_pretrained(final_adapter_dir)
processor.save_pretrained(OUTPUT_DIR)
print("saved:", final_adapter_dir)

del model
del trainer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 10) Reranker scoring helpers: A/B accuracy and cached tournament scores
def list_adapter_checkpoints():
    checkpoints = []
    for name in os.listdir(OUTPUT_DIR):
        path = os.path.join(OUTPUT_DIR, name)
        if name.startswith("checkpoint-") and os.path.exists(os.path.join(path, "adapter_config.json")):
            checkpoints.append(path)
    final_adapter = os.path.join(OUTPUT_DIR, "final_adapter")
    if os.path.exists(os.path.join(final_adapter, "adapter_config.json")):
        checkpoints.append(final_adapter)

    def step_key(path):
        match = re.search(r"checkpoint-(\d+)", os.path.basename(path))
        return int(match.group(1)) if match else 10**12

    return sorted(checkpoints, key=step_key)


def checkpoint_name(path):
    return os.path.basename(os.path.normpath(path))


def make_eval_pair_example(candidate_record, order_a, order_b, target="A"):
    return {
        "sample_id": candidate_record["sample_id"],
        "sentence": candidate_record["sentence"],
        "candidate_a": list(order_a),
        "candidate_b": list(order_b),
        "target": target,
    }


@torch.no_grad()
def score_ab_candidates(active_model, example, image_root):
    old_padding_side = processor.tokenizer.padding_side
    try:
        processor.tokenizer.padding_side = "right"
        messages, image_paths = reranker_messages(example, image_root, include_answer=False)
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in image_paths]
        inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        inputs = {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}
        outputs = active_model(**inputs)
        last_pos = int(inputs["attention_mask"][0].sum().item()) - 1
        logits = outputs.logits[0, last_pos]
        probs = torch.softmax(logits[[AB_TOKEN_IDS["A"], AB_TOKEN_IDS["B"]]].float(), dim=-1).detach().cpu().numpy()
        return {"A": float(probs[0]), "B": float(probs[1])}
    finally:
        processor.tokenizer.padding_side = old_padding_side


def structured_gap_bin(value):
    value = float(value)
    if value < 0.25:
        return "00_lt_0.25"
    if value < 0.75:
        return "01_0.25_0.75"
    if value < 1.5:
        return "02_0.75_1.5"
    return "03_ge_1.5"


def evaluate_ab_records(active_model, records, image_root, split_name, checkpoint):
    rows = []
    for record in tqdm(records, desc=f"A/B {checkpoint} {split_name}"):
        probs = score_ab_candidates(active_model, record, image_root)
        pred = "A" if probs["A"] >= probs["B"] else "B"
        gap = float(record.get("structured_gap_abs", abs(float(record.get("candidate_a_structured_score", 0.0)) - float(record.get("candidate_b_structured_score", 0.0)))))
        rows.append({
            "sample_id": record["sample_id"],
            "target": record["target"],
            "pred": pred,
            "correct": float(pred == record["target"]),
            "negative_type": record.get("negative_type", "unknown"),
            "comparison_type": record.get("comparison_type", "unknown"),
            "gold_rank": record.get("gold_rank"),
            "structured_gap_abs": gap,
            "structured_gap_bin": structured_gap_bin(gap),
            "negative_in_topk": record.get("negative_in_topk", None),
            "prob_a": probs["A"],
            "prob_b": probs["B"],
        })
    df = pd.DataFrame(rows)
    summary_rows = [{
        "checkpoint": checkpoint,
        "split": split_name,
        "group": "overall",
        "value": "overall",
        "accuracy": df["correct"].mean(),
        "count": len(df),
        "prediction_a_rate": float((df["pred"] == "A").mean()),
        "target_a_rate": float((df["target"] == "A").mean()),
    }]
    for group in ["negative_type", "comparison_type", "target", "gold_rank", "structured_gap_bin"]:
        for value, part in df.groupby(group, dropna=False):
            summary_rows.append({
                "checkpoint": checkpoint,
                "split": split_name,
                "group": group,
                "value": str(value),
                "accuracy": part["correct"].mean(),
                "count": len(part),
                "prediction_a_rate": float((part["pred"] == "A").mean()),
                "target_a_rate": float((part["target"] == "A").mean()),
            })
    return pd.DataFrame(summary_rows), df


def all24_softmax_scores(record, temperature=1.0):
    scores = np.array([float(item["structured_score"]) for item in record["all_candidates"]], dtype=np.float64)
    scores = scores / max(float(temperature), 1e-6)
    scores = scores - scores.max()
    probs = np.exp(scores)
    probs = probs / probs.sum()
    return {tuple(item["order"]): float(prob) for item, prob in zip(record["all_candidates"], probs)}


def standardize_scores(values):
    values = np.array(values, dtype=np.float64)
    std = float(values.std())
    if std < 1e-12:
        return np.zeros_like(values)
    return (values - float(values.mean())) / std


def build_pair_score_cache(active_model, candidate_records, image_root, top_k, checkpoint, split_name, force=False):
    path = os.path.join(EVAL_DIR, f"{checkpoint}_{split_name}_top{top_k}_pair_score_cache.jsonl")
    expected_ids = [record["sample_id"] for record in candidate_records]
    if os.path.exists(path) and not force:
        cached = read_jsonl(path)
        if [record["sample_id"] for record in cached] == expected_ids:
            print("[CACHE]", path)
            return cached
        print("Stale pair score cache, regenerating:", path)

    cache = []
    for record in tqdm(candidate_records, desc=f"pair scores {checkpoint} {split_name} top{top_k}"):
        orders = record["candidate_orders"][:top_k]
        n = len(orders)
        matrix = [[0.5 for _ in range(n)] for _ in range(n)]
        for i in range(n):
            for j in range(i + 1, n):
                probs_ab = score_ab_candidates(active_model, make_eval_pair_example(record, orders[i], orders[j]), image_root)
                probs_ba = score_ab_candidates(active_model, make_eval_pair_example(record, orders[j], orders[i]), image_root)
                p_i_beats_j = 0.5 * (float(probs_ab["A"]) + float(probs_ba["B"]))
                matrix[i][j] = p_i_beats_j
                matrix[j][i] = 1.0 - p_i_beats_j
        cache.append({"sample_id": record["sample_id"], "pair_matrix": matrix})
    write_jsonl(cache, path)
    return cache


def choose_from_pair_cache(record, pair_cache_record, top_k, lambda_value):
    orders = record["candidate_orders"][:top_k]
    softmax_lookup = all24_softmax_scores(record)
    structured_scores = np.array([softmax_lookup[tuple(order)] for order in orders], dtype=np.float64)
    if lambda_value >= 1.0 - 1e-12:
        reranker_scores = np.zeros(len(orders), dtype=np.float64)
        final_scores = structured_scores
        best_index = 0
        return orders[best_index], reranker_scores.tolist(), structured_scores.tolist(), final_scores.tolist()

    matrix = np.array(pair_cache_record["pair_matrix"], dtype=np.float64)[:top_k, :top_k]
    n = len(orders)
    reranker_scores = (matrix.sum(axis=1) - 0.5) / max(n - 1, 1)
    if STRUCTURED_SCORE_COMBINE_MODE == "zscore":
        structured_component = standardize_scores(structured_scores)
        reranker_component = standardize_scores(reranker_scores)
    else:
        structured_component = structured_scores
        reranker_component = reranker_scores
    final_scores = lambda_value * structured_component + (1.0 - lambda_value) * reranker_component
    best_index = int(np.argmax(final_scores))
    return orders[best_index], reranker_scores.tolist(), structured_scores.tolist(), final_scores.tolist()


def evaluate_from_pair_cache(candidate_records, pair_cache, top_k, lambda_value):
    rows = []
    cache_by_id = {record["sample_id"]: record for record in pair_cache} if pair_cache is not None else {}
    for record in candidate_records:
        gold = record.get("gold_order")
        if gold is None:
            continue
        cache_record = None if lambda_value >= 1.0 - 1e-12 else cache_by_id[record["sample_id"]]
        pred, reranker_scores, structured_scores, final_scores = choose_from_pair_cache(record, cache_record, top_k, lambda_value)
        structured_pred = record["candidate_orders"][0]
        metric = order_metric_row(pred, gold)
        structured_metric = order_metric_row(structured_pred, gold)
        gold_in_topk = gold in record["candidate_orders"][:top_k]
        rows.append({
            "sample_id": record["sample_id"],
            "gold_order": gold,
            "gold_rank": record.get("gold_rank"),
            "gold_in_topk": float(gold_in_topk),
            "structured_pred": structured_pred,
            "pred_order": pred,
            "top_k": top_k,
            "lambda": lambda_value,
            **metric,
            "structured_exact_match": structured_metric["exact_match"],
            "structured_pair_accuracy": structured_metric["pair_accuracy"],
            "structured_position_accuracy": structured_metric["position_accuracy"],
            "corrected_error": float(structured_metric["exact_match"] == 0.0 and metric["exact_match"] == 1.0),
            "introduced_error": float(structured_metric["exact_match"] == 1.0 and metric["exact_match"] == 0.0),
            "preserved_correct": float(structured_metric["exact_match"] == 1.0 and metric["exact_match"] == 1.0),
        })
    df = pd.DataFrame(rows)
    baseline_wrong_gold_available = ((df["structured_exact_match"] == 0.0) & (df["gold_in_topk"] == 1.0)).sum()
    baseline_correct = (df["structured_exact_match"] == 1.0).sum()
    summary = {
        "top_k": top_k,
        "lambda": lambda_value,
        "exact_match": df["exact_match"].mean(),
        "pair_accuracy": df["pair_accuracy"].mean(),
        "position_accuracy": df["position_accuracy"].mean(),
        "structured_exact_match": df["structured_exact_match"].mean(),
        "gold_recall_at_k": df["gold_in_topk"].mean(),
        "corrected_errors": int(df["corrected_error"].sum()),
        "introduced_errors": int(df["introduced_error"].sum()),
        "net_corrections": int(df["corrected_error"].sum() - df["introduced_error"].sum()),
        "rescue_rate": float(df["corrected_error"].sum() / baseline_wrong_gold_available) if baseline_wrong_gold_available else np.nan,
        "preservation_rate": float(df["preserved_correct"].sum() / baseline_correct) if baseline_correct else np.nan,
    }
    return summary, df


In [ ]:
# 11) Quick checkpoint evaluation with checkpoint deduplication
checkpoint_dirs = list_adapter_checkpoints()
print("checkpoints:", [checkpoint_name(path) for path in checkpoint_dirs])

quick_rows = []
quick_ab_summaries = []
name_to_dir = {checkpoint_name(path): path for path in checkpoint_dirs}

for checkpoint_dir in checkpoint_dirs:
    ckpt = checkpoint_name(checkpoint_dir)
    eval_model = load_adapter_model(checkpoint_dir, is_trainable=False)
    try:
        ab_summary, ab_rows = evaluate_ab_records(eval_model, quick_reranker_records, TRAIN_IMAGE_DIR, "quick50", ckpt)
        quick_ab_summaries.append(ab_summary)
        ab_rows.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_quick50_ab_predictions.csv"), index=False)
        pd.concat(quick_ab_summaries, ignore_index=True).to_csv(os.path.join(EVAL_DIR, "quick50_ab_accuracy.csv"), index=False)

        top_k = 5
        pair_cache = build_pair_score_cache(eval_model, quick_candidates, TRAIN_IMAGE_DIR, top_k=top_k, checkpoint=ckpt, split_name="quick50")
        for lambda_value in LAMBDAS:
            cache = None if lambda_value >= 1.0 - 1e-12 else pair_cache
            summary, _ = evaluate_from_pair_cache(quick_candidates, cache, top_k=top_k, lambda_value=lambda_value)
            summary.update({"checkpoint": ckpt, "split": "quick50"})
            quick_rows.append(summary)
            pd.DataFrame(quick_rows).to_csv(os.path.join(EVAL_DIR, "checkpoint_ranking_metrics_quick.csv"), index=False)
    finally:
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

quick_df = pd.DataFrame(quick_rows).sort_values(["exact_match", "net_corrections", "preservation_rate", "pair_accuracy"], ascending=False).reset_index(drop=True)
display(quick_df)
quick_non_structured = quick_df[quick_df["lambda"] < 1.0].copy()
if quick_non_structured.empty:
    raise ValueError("No lambda < 1.0 quick rows are available for reranker checkpoint selection.")
quick_ranked = quick_non_structured.drop_duplicates("checkpoint").head(3).reset_index(drop=True)
top_checkpoints = quick_ranked["checkpoint"].tolist()
print("top checkpoints from lambda<1.0 rows:", top_checkpoints)


In [ ]:
# 12) Select top-K/lambda on tuning150, evaluate selected config once on holdout150
tuning_rows = []
tuning_ab_summaries = []

for ckpt in top_checkpoints:
    checkpoint_dir = name_to_dir[ckpt]
    eval_model = load_adapter_model(checkpoint_dir, is_trainable=False)
    try:
        ab_summary, ab_rows = evaluate_ab_records(eval_model, tuning_reranker_records, TRAIN_IMAGE_DIR, "tuning150", ckpt)
        tuning_ab_summaries.append(ab_summary)
        ab_rows.to_csv(os.path.join(EVAL_DIR, f"{ckpt}_tuning150_ab_predictions.csv"), index=False)
        pd.concat(tuning_ab_summaries, ignore_index=True).to_csv(os.path.join(EVAL_DIR, "tuning150_ab_accuracy.csv"), index=False)

        max_top_k = max(RERANK_TOP_K_GRID)
        pair_cache = build_pair_score_cache(eval_model, tuning_candidates, TRAIN_IMAGE_DIR, top_k=max_top_k, checkpoint=ckpt, split_name="tuning150")
        for top_k in RERANK_TOP_K_GRID:
            for lambda_value in LAMBDAS:
                cache = None if lambda_value >= 1.0 - 1e-12 else pair_cache
                summary, _ = evaluate_from_pair_cache(tuning_candidates, cache, top_k=top_k, lambda_value=lambda_value)
                summary.update({"checkpoint": ckpt, "split": "tuning150"})
                tuning_rows.append(summary)
                pd.DataFrame(tuning_rows).to_csv(os.path.join(EVAL_DIR, "reranker_grid_tuning.csv"), index=False)
    finally:
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()

tuning_df = pd.DataFrame(tuning_rows).sort_values(["exact_match", "net_corrections", "preservation_rate", "pair_accuracy"], ascending=False).reset_index(drop=True)
display(tuning_df)

best = tuning_df.iloc[0].to_dict()
best_ckpt = best["checkpoint"]
best_checkpoint_dir = name_to_dir[best_ckpt]
best_top_k = int(best["top_k"])
best_lambda = float(best["lambda"])
print("BEST TUNING:", best)

eval_model = load_adapter_model(best_checkpoint_dir, is_trainable=False)
try:
    holdout_ab_summary, holdout_ab_rows = evaluate_ab_records(eval_model, holdout_reranker_records, TRAIN_IMAGE_DIR, "holdout150", best_ckpt)
    holdout_ab_summary.to_csv(os.path.join(EVAL_DIR, "holdout150_ab_accuracy.csv"), index=False)
    holdout_ab_rows.to_csv(os.path.join(EVAL_DIR, "holdout150_ab_predictions.csv"), index=False)

    holdout_pair_cache = None
    if best_lambda < 1.0 - 1e-12:
        holdout_pair_cache = build_pair_score_cache(eval_model, holdout_candidates, TRAIN_IMAGE_DIR, top_k=best_top_k, checkpoint=best_ckpt, split_name="holdout150")
    holdout_summary, holdout_predictions = evaluate_from_pair_cache(holdout_candidates, holdout_pair_cache, top_k=best_top_k, lambda_value=best_lambda)
    holdout_summary.update({"checkpoint": best_ckpt, "split": "holdout150"})
    holdout_predictions.to_csv(os.path.join(EVAL_DIR, "holdout_predictions.csv"), index=False)
    pd.DataFrame([holdout_summary]).to_csv(os.path.join(EVAL_DIR, "holdout_metrics.csv"), index=False)
finally:
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()

display(pd.DataFrame([holdout_summary]))

if os.path.exists(BEST_ADAPTER_DIR):
    shutil.rmtree(BEST_ADAPTER_DIR)
shutil.copytree(best_checkpoint_dir, BEST_ADAPTER_DIR)
processor.save_pretrained(BEST_ADAPTER_DIR)

best_config = {
    "experiment": "hard_negative_permutation_reranker_v2",
    "initial_adapter": INITIAL_ADAPTER_DIR,
    "reference_config": reference_config_summary,
    "candidate_generator": {
        "type": "pair_first_last_structured_decoder",
        "top_k": CANDIDATE_TOP_K,
        "alpha": STRUCTURED_ALPHA,
        "beta": STRUCTURED_BETA,
        "gamma": STRUCTURED_GAMMA,
        "pairwise_bidirectional": PAIRWISE_BIDIRECTIONAL,
        "min_pixels": MIN_PIXELS,
        "max_pixels": MAX_PIXELS,
        "prompt_version": CANDIDATE_PROMPT_VERSION,
    },
    "reranker": {
        "type": "ab_pairwise_preference_classification",
        "checkpoint": best_ckpt,
        "checkpoint_dir": BEST_ADAPTER_DIR,
        "source_checkpoint_dir": best_checkpoint_dir,
        "top_k": best_top_k,
        "lambda": best_lambda,
        "score_combine_mode": STRUCTURED_SCORE_COMBINE_MODE,
        "bidirectional_ab": True,
        "prompt_version": RERANKER_PROMPT_VERSION,
    },
    "negative_ratios": NEGATIVE_RATIOS,
    "better_wrong_min_gap": BETTER_WRONG_MIN_GAP,
    "record_stats": {
        "train": train_stats,
        "quick50": quick_stats,
        "tuning150": tuning_stats,
        "holdout150": holdout_stats,
    },
    "tuning": best,
    "holdout": holdout_summary,
    "baseline_test_score": 0.56544,
    "holdout_note": "This is the fixed validation partition from the previous Qwen2 experiment; treat as comparison holdout, not a fully independent unseen holdout if previously inspected.",
}
save_json(best_config, os.path.join(BEST_ADAPTER_DIR, "best_config.json"))
save_json(best_config, os.path.join(OUTPUT_DIR, "best_config.json"))
print(json.dumps(best_config, ensure_ascii=False, indent=2))


In [ ]:
# 13) Optional test inference and submission
test_candidates = generate_candidate_cache("test", test_df, TEST_IMAGE_DIR, has_gold=False)
with open(os.path.join(OUTPUT_DIR, "best_config.json"), "r", encoding="utf-8") as f:
    best_config = json.load(f)

test_top_k = int(best_config["reranker"]["top_k"])
test_lambda = float(best_config["reranker"]["lambda"])
test_checkpoint_dir = BEST_ADAPTER_DIR
test_ckpt = checkpoint_name(test_checkpoint_dir)

submission_rows = []
test_prediction_rows = []
if test_lambda >= 1.0 - 1e-12:
    for record in tqdm(test_candidates, desc="test choose structured-only"):
        pred, reranker_scores, structured_scores, final_scores = choose_from_pair_cache(record, None, test_top_k, test_lambda)
        submission_rows.append({"Id": record["sample_id"], "Answer": str(sequence_to_answer(pred))})
        test_prediction_rows.append({
            "sample_id": record["sample_id"],
            "pred_order": pred,
            "candidate_orders": record["candidate_orders"][:test_top_k],
            "structured_scores": structured_scores,
            "reranker_scores": reranker_scores,
            "final_scores": final_scores,
        })
else:
    test_model = load_adapter_model(test_checkpoint_dir, is_trainable=False)
    try:
        test_pair_cache = build_pair_score_cache(test_model, test_candidates, TEST_IMAGE_DIR, top_k=test_top_k, checkpoint=test_ckpt, split_name="test")
        cache_by_id = {record["sample_id"]: record for record in test_pair_cache}
        for record in tqdm(test_candidates, desc="test choose"):
            pred, reranker_scores, structured_scores, final_scores = choose_from_pair_cache(record, cache_by_id[record["sample_id"]], test_top_k, test_lambda)
            submission_rows.append({"Id": record["sample_id"], "Answer": str(sequence_to_answer(pred))})
            test_prediction_rows.append({
                "sample_id": record["sample_id"],
                "pred_order": pred,
                "candidate_orders": record["candidate_orders"][:test_top_k],
                "structured_scores": structured_scores,
                "reranker_scores": reranker_scores,
                "final_scores": final_scores,
            })
    finally:
        del test_model
        gc.collect()
        torch.cuda.empty_cache()

submission = pd.DataFrame(submission_rows)
if sample_submission_df is not None:
    sample_ids = sample_submission_df["Id"].astype(str).tolist()
    submission["Id"] = submission["Id"].astype(str)
    submission = submission.set_index("Id").loc[sample_ids].reset_index()
    assert len(submission) == len(sample_submission_df)
    assert submission["Id"].astype(str).tolist() == sample_ids
submission.to_csv(SUBMIT_PATH, index=False)
write_jsonl(test_prediction_rows, os.path.join(EVAL_DIR, "test_reranker_predictions.jsonl"))
shutil.copy2(SUBMIT_PATH, os.path.join(BEST_ADAPTER_DIR, "submission.csv"))
display(submission.head())
print("submission saved:", SUBMIT_PATH)


In [ ]:
# 14) Run config and artifact summary
run_config = {
    "experiment": "hard_negative_permutation_reranker_v2",
    "run_id": RUN_ID,
    "initial_adapter": INITIAL_ADAPTER_DIR,
    "reference_config": reference_config_summary,
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "split_dir": SPLIT_DIR,
    "split_hashes": {name: ids_hash(ids) for name, ids in split_ids.items()},
    "candidate_top_k": CANDIDATE_TOP_K,
    "train_top_k": TRAIN_TOP_K,
    "train_candidate_rows": TRAIN_CANDIDATE_ROWS,
    "batched_candidate_scoring": True,
    "negative_ratios": NEGATIVE_RATIOS,
    "max_records_per_sample": MAX_RECORDS_PER_SAMPLE,
    "better_wrong_pair_rate": BETTER_WRONG_PAIR_RATE,
    "better_wrong_min_gap": BETTER_WRONG_MIN_GAP,
    "structured_score_combine_mode": STRUCTURED_SCORE_COMBINE_MODE,
    "candidate_prompt_version": CANDIDATE_PROMPT_VERSION,
    "reranker_prompt_version": RERANKER_PROMPT_VERSION,
    "learning_rate": LEARNING_RATE,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "baseline_test_score": 0.56544,
    "shared_cache_root": SHARED_CACHE_ROOT,
    "output_dir": OUTPUT_DIR,
}
save_json(run_config, os.path.join(RUN_ROOT, "run_config.json"))
save_json(run_config, os.path.join(OUTPUT_DIR, "run_config.json"))
print(json.dumps(run_config, ensure_ascii=False, indent=2))


## Interpretation Notes

- `holdout150` is a comparison holdout within the previous Qwen2 validation partition. Do not describe it as a fully independent unseen holdout if it was already inspected during previous model development.
- Reranker maximum exact match is bounded by structured `recall@K`. Check `eval/topk_recall.csv` before spending time on long reranker training/evaluation.
- The training objective is A/B preference classification, not a margin-ranking loss.
- `lambda=1.0` is the structured-only fallback; `lambda=0.0` is reranker-only.
